# Phase Retrieval for Non-Uniform Filterbanks

## Tutorial: PGHI step by step

This notebook walks through phase gradient heap integration (PGHI) on a filterbank
whose channels do **not** share a hop size:

1. **Non-uniform filterbank analysis/synthesis** — how channels with different hop sizes tile the time-frequency plane
2. **Is it even a frame?** — the closed-form admissibility check, before you build anything
3. **The phase retrieval problem** — what is lost when you keep only magnitudes
4. **Phase gradient estimation** — the signal path (derivative filters) vs the magnitude path (log-magnitude ratios)
5. **Heap integration** — reconstructing phase from the gradient
6. **Refinement** — using PGHI as a Griffin-Lim initialisation

Everything here runs against the public `cool_frames` API.

> **Note on the differentiable variant.** Earlier revisions of this notebook demonstrated
> *Diff-RTPGHI*, the differentiable fixed-order variant from
> C. Hollomey, *Differentiable Real-Time Phase Reconstruction for Non-Uniform Filterbanks*.
> That algorithm is not part of the toolbox — it lives with the paper's own code — and
> `cool_frames.torch.phase.filterbankconstphase` is a NumPy shim that is explicitly **not**
> differentiable. For a phase-retrieval step that *can* sit inside a training graph, use
> `cool_frames.torch.phase.gla`; notebook 3 does exactly that.

## Setup

In [ ]:
# ── Setup ──
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                    'cool-frames @ git+https://github.com/allthatsounds/cool-frames.git'],
                   check=True)

import matplotlib.pyplot as plt

import numpy as np

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['figure.dpi'] = 100

In [ ]:
from cool_frames.filterbanks import filterbank, filterbankdual, filterbankresponse, ifilterbank
from cool_frames.filters import audfilters
from cool_frames.numpy.filterbanks._utils import normalise_a
from cool_frames.phase import filterbankconstphase, filterbankphasegrad, gla

print('All imports OK.')

## 1. Non-Uniform Filterbank Basics

A non-uniform filterbank has $M$ channels, each with its own hop size $a_m$ and centre
frequency $f_{c,m}$. Low-frequency channels have large hop sizes (coarse time resolution,
fine frequency resolution) and high-frequency channels have small hop sizes (fine time
resolution, coarser frequency resolution).

`audfilters` is one of several designers that place channels this way — `cqtfilters`,
`waveletfilters`, `greenwoodfilters`, `gabfilters` and `warpedfilters` all produce the same
kind of object and everything below works on any of them. Swap the designer in the next
cell and the rest of the notebook is unchanged.

In [ ]:
fs = 16000  # Sample rate
Ls = 8000   # Signal length (0.5 seconds)
redmul = 8.0  # Redundancy multiplier

# Design the filterbank.  Every designer returns the same 5-tuple:
#   g    – list of M filter dicts
#   a    – hop sizes (integer or rational, one per channel)
#   fc   – centre frequencies in Hz
#   L    – the DFT length the bank is built for
#   info – designer-specific geometry (spacing, bandwidths, ...)
g, a, fc_hz, L, info = audfilters(fs, Ls, redmul=redmul)
M = len(g)
a_norm = normalise_a(a, M)
a_int = np.array([int(a_norm[m, 0]) for m in range(M)])

print(f'Filterbank: {M} channels, signal length L={L}')
print(f'Hop sizes: min={min(a_int)}, max={max(a_int)}')
print(f'Centre frequencies: {fc_hz[0]:.0f} Hz to {fc_hz[-1]:.0f} Hz')

# Number of frames per channel
N = [L // a_int[m] for m in range(M)]
redundancy = sum(N) / L
print(f'Redundancy: {redundancy:.1f}x')

In [ ]:
# Visualise hop sizes and frame counts across channels
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(range(M), a_int, color='steelblue', alpha=0.7)
ax1.set_xlabel('Channel index')
ax1.set_ylabel('Hop size (samples)')
ax1.set_title('Hop size per channel')

ax2.bar(range(M), N, color='coral', alpha=0.7)
ax2.set_xlabel('Channel index')
ax2.set_ylabel('Number of frames')
ax2.set_title('Frames per channel')

plt.tight_layout()
plt.show()

## 2. Is it a frame? The admissibility floor

Before inverting anything it is worth asking whether the bank *can* be inverted. A painless
filterbank is a frame with bounds $0 < A \le B < \infty$; if $A = 0$ some part of the
spectrum is annihilated and no dual exists, however well conditioned the rest looks.

$A = 0$ happens exactly when the filters leave a DFT bin uncovered, which is integer
arithmetic on the design parameters — so it can be predicted *from the parameters alone*,
before any filter is built. `predict_admissible` does that in closed form; the measured
frame response is the ground truth it was validated against.

In [ ]:
# Every designer runs the check itself and publishes the verdict.
pred = info['admissible']
print(f"predicted frame: {pred['is_frame']}   (rho = {pred['rho']:.3f}, "
      f"kappa_pred = {pred['kappa_pred']:.2f})")
if not pred['is_frame']:
    print(f"  first uncovered bin: {pred['first_hole_bin']} "
          f"({pred['n_hole_bins']} bins in total)")

# Ground truth: the measured frame response.  Its minimum over the spectrum is A.
R = np.real(np.asarray(filterbankresponse(g, a_norm, L, real=True)))
print(f'measured  A = {R.min():.4e},  B = {R.max():.4e},  '
      f'kappa = {R.max()/max(R.min(), 1e-300):.2f}')
print(f'measured frame: {R.min() > 1e-12 * R.max()}')

Because the check runs inside the designer, asking for a bank that cannot work tells you so
immediately — and tells you *which* band is missing, rather than leaving you to discover it
when the dual comes back all zeros.

In [ ]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    g_bad, a_bad, fc_bad, L_bad, info_bad = audfilters(fs, 1024, M=8)

for w in caught:
    print(f'{w.category.__name__}: {w.message}')

In [ ]:
# What the floor looks like: sweep the number of channels.  The predictor runs
# from the parameters, so this costs nothing beyond designing the banks.
Ms = np.arange(4, 40)
ok = []
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for M_try in Ms:
        try:
            _g, _a, _fc, _L, _info = audfilters(fs, 1024, M=int(M_try))
        except Exception:
            ok.append(np.nan)
            continue
        ok.append(1.0 if _info['admissible']['is_frame'] else 0.0)

fig, ax = plt.subplots(figsize=(10, 2.6))
ax.step(Ms, ok, where='mid', color='steelblue')
ax.fill_between(Ms, 0, ok, step='mid', alpha=0.25, color='steelblue')
ax.set_xlabel('Number of channels M')
ax.set_yticks([0, 1])
ax.set_yticklabels(['A = 0', 'frame'])
ax.set_title('Admissibility floor, predicted from the design parameters alone')
plt.tight_layout()
plt.show()

## 3. Analysis and Perfect Reconstruction

With the canonical dual we can perfectly reconstruct a signal from its complex coefficients
(magnitude **and** phase). Let's verify this.

In [ ]:
# Create a test signal: sum of sinusoids
t = np.arange(Ls) / fs
sig = 0.5 * np.sin(2*np.pi*440*t) + 0.3 * np.sin(2*np.pi*1000*t) + 0.2 * np.sin(2*np.pi*2500*t)
sig = sig / np.max(np.abs(sig)) * 0.9

# Pad to filterbank length L
sig_padded = np.zeros(L)
sig_padded[:Ls] = sig

# Analysis: get complex coefficients
c = filterbank(sig_padded, g, a_norm, L=L)
print(f'Got {M} channels, shapes: {[np.asarray(ci).shape for ci in c[:3]]}...')

# Synthesis: reconstruct from complex coefficients.
# filterbankdual(..., real=True) is the dual of the real-signal (single-sided) bank.
gd = filterbankdual(g, a_norm, L, real=True)
sig_recon = ifilterbank(c, gd, a_norm, Ls=L, real=True)
sig_recon = np.real(sig_recon[:Ls])

# Measure reconstruction error
err = np.max(np.abs(sig - sig_recon))
print(f'Perfect reconstruction error: {err:.2e} (should be ~1e-14)')

## 4. The Phase Retrieval Problem

Now suppose we only have the **magnitudes** — the phases are lost. This happens in many
audio pipelines (spectral modification, neural network prediction). How well can we
reconstruct the signal from magnitudes alone?

In [ ]:
# Discard phases, keep only magnitudes
s_list = [np.abs(np.asarray(ci).ravel()) for ci in c]

# Reconstruct with zero phase (worst case)
c_zero = [s.astype(complex) for s in s_list]
sig_zero = ifilterbank(c_zero, gd, a_norm, Ls=L, real=True)
sig_zero = np.real(sig_zero[:Ls])


def sdr(ref, est):
    n = min(len(ref), len(est))
    r, e = ref[:n], est[:n]
    return 10 * np.log10(np.sum(r**2) / (np.sum((r - e)**2) + 1e-30))


def aligned_sdr(ref, c_recon, n_angles=4096):
    # A global phase rotation is unobservable in the magnitudes, so any phase
    # retrieval method may return the signal rotated.  Compare against the best
    # rotation, or the number says more about the rotation than the method.
    # Synthesis is real-linear in the coefficients, so
    #     y(theta) = cos(theta) * y(c) + sin(theta) * y(i c)
    # holds exactly and two syntheses cover every rotation.
    def syn(cc):
        return np.real(ifilterbank(cc, gd, a_norm, Ls=L, real=True))[:Ls]
    u, v = syn(c_recon), syn([1j * ci for ci in c_recon])
    r = np.asarray(ref[:Ls], float)
    th = np.linspace(0, 2*np.pi, n_angles, endpoint=False)
    ct, st = np.cos(th), np.sin(th)
    err = (r @ r) - 2*(ct*(r @ u) + st*(r @ v)) \
        + ct**2*(u @ u) + 2*ct*st*(u @ v) + st**2*(v @ v)
    return float(10 * np.log10((r @ r) / max(err.min(), 1e-30)))


def sc_roundtrip(c_target, c_recon):
    # Round-trip spectral convergence: synthesise, re-analyse, compare magnitudes.
    sig_r = np.real(ifilterbank(c_recon, gd, a_norm, Ls=L, real=True))
    c_re = filterbank(sig_r, g, a_norm, L=L)
    r = np.concatenate([np.abs(np.asarray(ci).ravel()) for ci in c_target])
    e = np.concatenate([np.abs(np.asarray(ci).ravel()) for ci in c_re])
    return 20 * np.log10(np.linalg.norm(r - e) / (np.linalg.norm(r) + 1e-30))


print(f'Zero-phase SDR: {aligned_sdr(sig, c_zero):.1f} dB')
print(f'Zero-phase SC:  {sc_roundtrip(c, c_zero):.1f} dB')
print('0 dB SDR means the error is as large as the signal — nothing survives.')

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t[:2000], sig[:2000], label='Original', alpha=0.8)
ax.plot(t[:2000], sig_zero[:2000], label='Zero phase', alpha=0.6)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.legend()
ax.set_title('Phase matters! Zero-phase reconstruction is useless.')
plt.tight_layout()
plt.show()

## 5. Phase Gradient Estimation

PGHI recovers the phase by integrating the **phase gradient** — the rate of change of phase
in time (instantaneous frequency) and in frequency (group delay). There are two ways to get
that gradient, and the difference between them is the whole story.

### 5a. The signal path — derivative filters
Convolve the signal with the frequency-derivative of each filter. This gives the absolute
instantaneous frequency directly and is accurate, but it needs the signal, not just its
magnitudes. `filterbankphasegrad` does this.

### 5b. The magnitude path — log-magnitude ratios
Estimate the gradient from cross-channel and cross-frame log-magnitude differences, via the
Cauchy–Riemann relations for a Gaussian window. This needs *only* the magnitudes, which is
the case that matters when the magnitudes came out of a network. It is also the harder case:
the relation involves a per-channel time–frequency ratio $\gamma$ whose convention is
designer-specific.

In [ ]:
# 5a. Signal path: derivative-filter gradients
tgrad_l, fgrad_l, s_l, c_with_grad = filterbankphasegrad(sig_padded, g, a_norm, L)

# True phase gradients (from the actual complex coefficients)
true_phases = [np.angle(np.asarray(ci).ravel()) for ci in c]

ch = M // 2  # Middle channel
true_if = np.diff(np.unwrap(true_phases[ch])) / a_int[ch]  # True inst. freq
est_if = np.asarray(tgrad_l[ch]).ravel() * np.pi           # Derivative-filter estimate

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(true_if[:200], label='True IF', alpha=0.8)
ax.plot(est_if[:200], label='Derivative-filter estimate', alpha=0.6, linestyle='--')
ax.set_xlabel('Frame index')
ax.set_ylabel('Instantaneous frequency (rad/sample)')
ax.set_title(f'Channel {ch}: derivative-filter gradient vs true IF')
ax.legend()
plt.tight_layout()
plt.show()

n_compare = min(len(true_if), len(est_if))
if_err = np.abs(true_if[:n_compare] - est_if[:n_compare])
print(f'Channel {ch}: mean IF error = {np.mean(if_err):.4f} rad/sample')

### The `sqtfr` convention

On the magnitude path, `filterbankconstphase` needs `sqtfr` — the square root of the
per-channel time–frequency ratio. **The convention is designer-specific**, and getting it
wrong costs accuracy silently rather than raising:

| designer | `sqtfr` |
|---|---|
| `audfilters` | `np.ones(M)` — the ERB bandwidth scaling already absorbs $\gamma^2$ into the hop sizes |
| `waveletfilters` | `np.sqrt([g[m]['tfr'](L) for m in range(M)])` |
| `gabfilters` | $\gamma = C_g\,g_l^2$, with $C_g^{\mathrm{hann}} = 0.25645$ |
| `cqtfilters` | not established |

In particular, do **not** reach for `compute_tfr_from_filters` on an `audfilters` bank: it
returns $L/\gamma$, the support length, which is a different quantity. None of these
conventions has yet been validated side by side against MATLAB LTFAT, and that check is
what currently limits magnitude-only phase retrieval on these banks — see the limitations
section of the toolbox paper.

In [ ]:
# 5b. Magnitude path: the audfilters convention
sqtfr = np.ones(M)
print('sqtfr =', sqtfr[:5], '...  (audfilters convention)')

## 6. Heap Integration

PGHI integrates the gradient by walking the time-frequency plane in descending magnitude
order, starting from the loudest coefficient: the phase of a reliable coefficient is used to
predict its neighbours', so error accumulates along paths of high energy rather than
uniformly. Coefficients below `tol` of the peak get random phase, since integrating through
noise only spreads it.

`filterbankconstphase` runs this on both paths. The calling convention selects which:

In [ ]:
# Signal path — filters and hops, gradients computed internally
c_sig, used_sig = filterbankconstphase(sig_padded, g, a_norm, L, fc_hz, tol=1e-6)

# Magnitude path — magnitudes, hop sizes, centre frequencies, and sqtfr
c_mag, used_mag = filterbankconstphase(s_list, a_int, np.asarray(fc_hz, float),
                                       sqtfr=sqtfr, fs=fs, tol=1e-6, rng=0)

for name, cc in (('PGHI, signal path   ', c_sig),
                 ('PGHI, magnitude path', c_mag),
                 ('zero phase          ', c_zero)):
    print(f'{name}: SC = {sc_roundtrip(c, cc):6.1f} dB    '
          f'SDR = {aligned_sdr(sig, cc):6.1f} dB')

In [ ]:
sig_pghi = np.real(ifilterbank(c_sig, gd, a_norm, Ls=L, real=True))[:Ls]

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t[:2000], sig[:2000], label='Original', alpha=0.8)
ax.plot(t[:2000], sig_pghi[:2000], label='PGHI (signal path)', alpha=0.6, linestyle='--')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.legend()
ax.set_title('PGHI reconstruction')
plt.tight_layout()
plt.show()

Two things to read off that table.

First, the gap between the two paths is the honest open problem here. The signal path has
the true instantaneous frequency; the magnitude path infers it from a Gaussian
approximation whose $\gamma$ convention is unvalidated against MATLAB LTFAT. Closing that
gap is what the `sqtfr` table above is waiting on.

Second, the two metrics disagree about the magnitude path, and that disagreement is
informative rather than a bug. Spectral convergence says it is clearly better than zero
phase (it reproduces the magnitudes far better); aligned SDR says it is slightly worse
(the waveform it produces is not the original waveform). Both are true: the magnitude path
finds *a* signal with approximately the right magnitudes, which is what phase retrieval is
allowed to do, but not the one we started from. SC is the metric to trust for phase
retrieval quality; SDR is only meaningful once the estimate is close enough that there is
one plausible answer.

## 7. Improving Quality with fGLA

PGHI gives a single-pass estimate. Feeding it to the fast Griffin-Lim algorithm as an
initialisation gets closer still, at the cost of iterating.

In [ ]:
# fGLA from the PGHI estimate ('input' takes the phase of what you pass in)
c_fgla, f_fgla, relres, niter = gla(
    c_sig, g, a_norm, L=L, real=True,
    maxit=100, method='fgla', startphase='input'
)

# fGLA from zero phase, for comparison
c_fgla0, _, _, _ = gla(
    s_list, g, a_norm, L=L, real=True,
    maxit=100, method='fgla', startphase='zero'
)

print(f'fGLA (PGHI init, 100 it.) SC: {sc_roundtrip(c, c_fgla):6.1f} dB')
print(f'fGLA (zero init, 100 it.) SC: {sc_roundtrip(c, c_fgla0):6.1f} dB')
print(f'PGHI alone (signal path)  SC: {sc_roundtrip(c, c_sig):6.1f} dB')
print()
print('fGLA is better but takes 100 iterations and is not causal.')
print('Note that `relres` here is the RTISIL-family residual, not a convergence measure —')
print(f'it reports {relres[-1]:.4g} after {niter} iterations, which is not an error bound.')

## Summary

| Property | PGHI (signal path) | PGHI (magnitude path) | fGLA |
|----------|-----|-----|------|
| Single-pass | ✓ | ✓ | ✗ (iterative) |
| Needs only magnitudes | ✗ | ✓ | ✓ |
| Causal / streaming | ✓ | ✓ | ✗ |
| Differentiable | ✗ | ✗ | ✓ (torch backend) |

The magnitude path is the one that matters for generative pipelines, and it is the one still
limited by the unvalidated $\gamma$ convention.

See **Notebook 2** for a comparison across the phase-retrieval methods the toolbox ships,
and **Notebook 3** for training a network through a differentiable phase-retrieval step.